This Model is Done By Hosny Hassan, I hope this model helps you in a way or another. I would like to learn more, so if you have any recommendation, I would be happy to here

In [ ]:
import os         # Importing necessary libraries #
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
DATA_DIR =  r"" # Make sure that your folder has two folders, each containing the respective class images, as they will be used for labeling #
MODEL_PATH = "Lung_Carcinoma_ct_classifier.pth"
BATCH_SIZE = 8
IMG_SIZE = 224
EPOCHS = 5 # You can increase this for better performance, but it will take more time to train #

In [32]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                                          [0.229, 0.224, 0.225])
])

In [33]:
train_dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [34]:
class_names = train_dataset.classes
print("Classes:", class_names)

Classes: ['Healthy Lung CT', 'Tumorrized Lung']


In [35]:
# Use pre-trained ResNet18 for better performance
model = models.resnet18(pretrained=True)
# Freeze all layers except the last few
for param in model.parameters():
    param.requires_grad = False
# Unfreeze the layer4 and fc
for param in model.layer4.parameters():
    param.requires_grad = True
# Replace the final layer
model.fc = nn.Linear(model.fc.in_features, 2)

f:\Work\Python Learning\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
f:\Work\Python Learning\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# Use this if you have a powerful GPU, otherwise it will run on CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
model = model.to(device)
device

device(type='cpu')

In [37]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [38]:
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / len(train_dataset) * 100
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}%")

Epoch 1/5 - Loss: 0.2759, Acc: 89.86%
Epoch 2/5 - Loss: 0.1905, Acc: 93.84%
Epoch 3/5 - Loss: 0.1380, Acc: 95.61%
Epoch 4/5 - Loss: 0.1253, Acc: 96.32%
Epoch 5/5 - Loss: 0.0982, Acc: 96.88%


In [39]:
from PIL import Image

In [ ]:
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
image_path = r"" # Put the image path of your image that you want to evaluate to test whether it works correctly #
image = Image.open(image_path).convert('RGB')
image = transform(image).unsqueeze(0).to(device)
with torch.no_grad():
    outputs = model(image)
    print("Outputs:", outputs)
    _, predicted = torch.max(outputs, 1)
    print("Predicted index:", predicted.item())
    predicted_class = class_names[predicted.item()]

print(f"Predicted: {predicted_class}")
print("Class names:", class_names)

Outputs: tensor([[-6.3897,  6.1053]])
Predicted index: 1
Predicted: Tumorrized Lung
Class names: ['Healthy Lung CT', 'Tumorrized Lung']


In [48]:
torch.save({
    "model_state": model.state_dict(),
    "class_names": class_names
}, MODEL_PATH)

print(" Model saved at", MODEL_PATH)

 Model saved at Lung_Carcinoma_ct_classifier.pth


Now, After everything was done, we want to use the model with the learned parameters without relearning again by just load the model structure and parameters

Now, Any time you need to use the model, just run the cell below, but donot forget to put the image path

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# Load the pre-trained ResNet18 model [Load the model architecture first, then load the weights]
model = models.resnet18(pretrained=True)
# Freeze all layers except the last few
for param in model.parameters():
    param.requires_grad = False
# Unfreeze the layer4 and fc
for param in model.layer4.parameters():
    param.requires_grad = True
# Replace the final layer
model.fc = nn.Linear(model.fc.in_features, 2)

# Load the saved model
MODEL_PATH = "Lung_Carcinoma_ct_classifier.pth"
checkpoint = torch.load(MODEL_PATH)
model.load_state_dict(checkpoint["model_state"])
class_names = checkpoint["class_names"]
model.eval()

# Specify the image path here
image_path = r""  # Change this to the image path you want to use

# Prepare and predict on the image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
image = Image.open(image_path).convert('RGB')
image = transform(image).unsqueeze(0)
with torch.no_grad():
    outputs = model(image)
    print("Outputs:", outputs)
    _, predicted = torch.max(outputs, 1)
    print("Predicted index:", predicted.item())
    predicted_class = class_names[predicted.item()]

print(f"Predicted: {predicted_class}")
print("Class names:", class_names)

Outputs: tensor([[-4.4465,  4.4977]])
Predicted index: 1
Predicted: Tumorrized Lung
Class names: ['Healthy Lung CT', 'Tumorrized Lung']
